<a href="https://colab.research.google.com/github/sudikshyapant/sae_probing/blob/main/sae_probing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAE Probing: Athlete Sport Football

Re-implementation of the SAE probing strategy from *Are Sparse Autoencoders Useful? A Case Study in Sparse Probing* (Kantamneni et al., 2025).

**Dataset:** `154_athlete_sport_football` — given an athlete name, classify whether they play football (1) or not (0).

**Pipeline:**
1. Load dataset & split
2. Extract last-token hidden states from Gemma-2-2B at layer 20
3. Encode activations through GemmaScope SAE → sparse latents Z
4. Select top-k SAE features by mean activation difference (k=16, k=128)
5. Train logistic regression probe on Z[:,I_k], select C via val set
6. Evaluate on held-out test set

In [10]:
# !pip install torch transformers sae-lens scikit-learn datasets numpy pandas tqdm huggingface-hub accelerate

In [11]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sae_lens import SAE

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [18]:
_base = Path("/content") if os.path.exists("/content") else Path.cwd()

CONFIG = {
    "model_name":   "google/gemma-2-2b",
    "target_layer": 20,
    "batch_size":   8,
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
    "sae_release":  "gemma-scope-2b-pt-res-canonical",
    "sae_id":       "layer_20/width_16k/canonical", # Corrected SAE ID
    "k_values":     [16, 128],
    "C_values":     list(np.logspace(-4, 4, 9)),
    "data_url":    "https://raw.githubusercontent.com/sudikshyapant/sae_probing/main/data/154_athlete_sport_football.csv",
    "test_size":    0.4,
    "val_size":     0.2,
    "random_state": 42,
    "cache_dir":    _base / "cache",
    "results_dir":  _base / "result",
}

Path(CONFIG["cache_dir"]).mkdir(exist_ok=True)
Path(CONFIG["results_dir"]).mkdir(exist_ok=True)
print("Device:", CONFIG["device"])

Device: cuda


---
## Step 1 — Load Dataset & Split

- **80% train_pool / 20% test** (stratified on label)
- From train_pool: **80% train / 20% val** (val used only for C selection)

In [13]:
df = pd.read_csv(CONFIG["data_url"])
print(f"Total examples: {len(df)}")
print(df["target"].value_counts().rename({0: "not football", 1: "football"}))

prompts = df["prompt"].tolist()
labels  = df["target"].tolist()

X_trainpool, X_test, y_trainpool, y_test = train_test_split(
    prompts, labels,
    test_size=CONFIG["test_size"],
    stratify=labels,
    random_state=CONFIG["random_state"],
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainpool, y_trainpool,
    test_size=CONFIG["val_size"],
    stratify=y_trainpool,
    random_state=CONFIG["random_state"],
)

y_train = np.array(y_train)
y_val   = np.array(y_val)
y_test  = np.array(y_test)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Total examples: 1772
target
not football    886
football        886
Name: count, dtype: int64
Train: 850, Val: 213, Test: 709


---
## Step 2 — Extract Activations (`X_l`)

Run each prompt through Gemma-2-2B and record the **last-token hidden state** at layer 20.
Cached to `cache/activations.pt` so the model only runs once.

In [14]:
def load_model_and_tokenizer(config):
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
        print("Authenticated with HuggingFace via Colab secret.")
    except Exception:
        pass  # Not in Colab or token not set — rely on local login

    print(f"Loading tokenizer: {config['model_name']}")
    tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
    print(f"Loading tokenizer: {config['model_name']}")
    tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model (this may take a while)...")
    model = AutoModelForCausalLM.from_pretrained(
        config["model_name"],
        output_hidden_states=True,
        torch_dtype=torch.float16 if config["device"] == "cuda" else torch.float32,
    ).to(config["device"])
    model.eval()
    print("Model loaded.")
    return model, tokenizer


def extract_activations(prompts, model, tokenizer, layer_idx, batch_size, device):
    """Extract last-token hidden state at layer_idx for each prompt.
    Returns np.ndarray of shape (n_prompts, d_model).
    """
    all_activations = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="Extracting activations"):
        batch = prompts[i : i + batch_size]

        # Dynamic padding — no fixed max_seq_len needed for short athlete names
        tokens = tokenizer(batch, return_tensors="pt", padding=True, truncation=False).to(device)

        with torch.no_grad():
            out = model(**tokens)

        # hidden_states[0]=embedding, hidden_states[i]=layer i output
        hidden = out.hidden_states[layer_idx]            # (B, seq_len, d_model)
        seq_lens = tokens["attention_mask"].sum(dim=1) - 1
        last_tok = hidden[torch.arange(len(batch)), seq_lens]  # (B, d_model)

        all_activations.append(last_tok.float().cpu().numpy())

    return np.concatenate(all_activations, axis=0)

In [15]:
CACHE_ACT = Path(CONFIG["cache_dir"]) / "activations.pt"

if CACHE_ACT.exists():
    print("Loading cached activations...")
    act_cache   = torch.load(CACHE_ACT, weights_only=False)
    X_act_train = act_cache["train"]
    X_act_val   = act_cache["val"]
    X_act_test  = act_cache["test"]
    print(f"Loaded. Train shape: {X_act_train.shape}")
else:
    model, tokenizer = load_model_and_tokenizer(CONFIG)

    X_act_train = extract_activations(X_train, model, tokenizer, CONFIG["target_layer"], CONFIG["batch_size"], CONFIG["device"])
    X_act_val   = extract_activations(X_val,   model, tokenizer, CONFIG["target_layer"], CONFIG["batch_size"], CONFIG["device"])
    X_act_test  = extract_activations(X_test,  model, tokenizer, CONFIG["target_layer"], CONFIG["batch_size"], CONFIG["device"])

    torch.save({"train": X_act_train, "val": X_act_val, "test": X_act_test}, CACHE_ACT)
    print(f"Activations saved. Train shape: {X_act_train.shape}")

    del model
    if CONFIG["device"] == "cuda":
        torch.cuda.empty_cache()

Authenticated with HuggingFace via Colab secret.
Loading tokenizer: google/gemma-2-2b
Loading tokenizer: google/gemma-2-2b
Loading model (this may take a while)...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Model loaded.


Extracting activations:   0%|          | 0/107 [00:00<?, ?it/s]

Extracting activations:   0%|          | 0/27 [00:00<?, ?it/s]

Extracting activations:   0%|          | 0/89 [00:00<?, ?it/s]

Activations saved. Train shape: (850, 2304)


---
## Step 3 — Encode Through SAE → Latents (`Z`)

**SAELens** is a library that makes it easy to load pre-trained Sparse Autoencoders.
- `sae_release` = the HuggingFace repo name for the SAE collection (`gemma-scope-2b-pt-res-canonical`)
- `sae_id` = the specific SAE within that collection (`layer_20/width_16k`)

The SAE maps each dense activation (dim=2304) to a sparse vector (dim=16,384),
where ~71 entries are nonzero on average (hence `average_l0_71`).

Cached to `cache/latents.pt`.

In [16]:
def encode_with_sae(activations, sae, batch_size, device):
    """Encode dense activations → sparse SAE latents.
    Returns np.ndarray of shape (n_prompts, sae_width).
    """
    all_latents = []

    for i in tqdm(range(0, len(activations), batch_size), desc="SAE encoding"):
        batch = torch.tensor(activations[i : i + batch_size], dtype=torch.float32).to(device)
        with torch.no_grad():
            latents = sae.encode(batch)  # (B, sae_width)
        all_latents.append(latents.cpu().numpy())

    return np.concatenate(all_latents, axis=0)

In [20]:
CACHE_LAT = Path(CONFIG["cache_dir"]) / "latents.pt"

if CACHE_LAT.exists():
    print("Loading cached SAE latents...")
    lat_cache = torch.load(CACHE_LAT, weights_only=False)
    Z_train   = lat_cache["train"]
    Z_val     = lat_cache["val"]
    Z_test    = lat_cache["test"]
    print(f"Loaded. Train shape: {Z_train.shape}")
else:
    print(f"Loading SAE: {CONFIG['sae_release']} / {CONFIG['sae_id']}")
    sae, cfg_dict, _ = SAE.from_pretrained_with_cfg_and_sparsity(
        release=CONFIG["sae_release"],
        sae_id=CONFIG["sae_id"],
        device=CONFIG["device"],
    )
    sae.eval()
    print(f"SAE loaded. Width: {cfg_dict.get('d_sae', '?')}")

    Z_train = encode_with_sae(X_act_train, sae, CONFIG["batch_size"], CONFIG["device"])
    Z_val   = encode_with_sae(X_act_val,   sae, CONFIG["batch_size"], CONFIG["device"])
    Z_test  = encode_with_sae(X_act_test,  sae, CONFIG["batch_size"], CONFIG["device"])

    torch.save({"train": Z_train, "val": Z_val, "test": Z_test}, CACHE_LAT)
    print(f"Latents saved. Train shape: {Z_train.shape}")

    del sae
    if CONFIG["device"] == "cuda":
        torch.cuda.empty_cache()

Loading cached SAE latents...
Loaded. Train shape: (850, 16384)


---
## Step 4 — Feature Selection (`I_k`)

For each k in `[16, 128]`, select the top-k SAE latent indices with the largest
mean activation difference between classes (Equation 1 from the paper):

$$\Delta_j = |\bar{Z}_j^{(1)} - \bar{Z}_j^{(0)}|$$

In [21]:
def select_top_k_latents(Z_train, y_train, k):
    """Return indices of k SAE latents with largest mean activation difference."""
    mean_class1 = Z_train[y_train == 1].mean(axis=0)
    mean_class0 = Z_train[y_train == 0].mean(axis=0)
    delta       = np.abs(mean_class1 - mean_class0)
    return np.argsort(delta)[-k:]


feature_indices = {}
for k in CONFIG["k_values"]:
    feature_indices[k] = select_top_k_latents(Z_train, y_train, k)
    print(f"k={k:4d}: top feature indices (first 5): {feature_indices[k][:5]}")

k=  16: top feature indices (first 5): [10608 13409 12108 10111  8000]
k= 128: top feature indices (first 5): [ 7071  1188  9012 15403 14051]


---
## Step 5 — Train Probe & Evaluate

For each k:
1. Slice latents to `Z[:, I_k]`
2. Grid-search C over `[1e-4 ... 1e4]` using val AUC
3. Report test AUC with best C

In [29]:
def train_probe(Z_tr, y_tr, Z_va, y_va, Z_te, y_te, C_values):
    """Logistic regression probe with val-based C selection."""
    best_C, best_val_auc, best_model = None, -1, None

    for C in C_values:
        clf = LogisticRegression(C=C, max_iter=10000, solver="lbfgs", random_state=42)
        clf.fit(Z_tr, y_tr)
        val_auc = roc_auc_score(y_va, clf.predict_proba(Z_va)[:, 1])
        if val_auc > best_val_auc:
            best_val_auc, best_C, best_model = val_auc, C, clf

    test_auc = roc_auc_score(y_te, best_model.predict_proba(Z_te)[:, 1])
    return {"best_C": best_C, "val_auc": best_val_auc, "test_auc": test_auc, "model": best_model}


results = {}
for k in CONFIG["k_values"]:
    idx = feature_indices[k]
    res = train_probe(
        Z_train[:, idx], y_train,
        Z_val[:,   idx], y_val,
        Z_test[:,  idx], y_test,
        CONFIG["C_values"],
    )
    results[f"sae_k{k}"] = res
    print(f"k={k:4d} | best C={res['best_C']:.0e} | val AUC={res['val_auc']:.4f} | test AUC={res['test_auc']:.4f}")

k=  16 | best C=1e-02 | val AUC=0.5599 | test AUC=0.5098
k= 128 | best C=1e-01 | val AUC=0.7366 | test AUC=0.6625


---
## Results

In [30]:
results_path = Path(CONFIG["results_dir"]) / "probe_model.pkl"
with open(results_path, "wb") as f:
    pickle.dump({"config": CONFIG, "results": results, "feature_indices": feature_indices}, f)
print(f"Results saved to {results_path}")

rows = [
    {"Method": m, "Best C": r["best_C"], "Val AUC": round(r["val_auc"], 4), "Test AUC": round(r["test_auc"], 4)}
    for m, r in results.items()
]
print(pd.DataFrame(rows).set_index("Method").to_string())

Results saved to /content/result/probe_model.pkl
          Best C  Val AUC  Test AUC
Method                             
sae_k16     0.01   0.5599    0.5098
sae_k128    0.10   0.7366    0.6625
